In [2]:
import sys
sys.path.append("..")

import h5py
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from utils.das_dataset import DasDataset
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from utils.noise_scheduler import LinearNoiseScheduler 
from IPython.display import clear_output

In [3]:
das_data = Path("/datasets/training/earthquake/share/SISSLE_south30")
train_files_noise = [das_data/"train_1000.h5"]#sorted(das_data.glob("train_*.h5"))
test_files_noise = [das_data/"test_1000.h5"]#sorted(das_data.glob("test_*.h5"))

device= "cuda" if torch.cuda.is_available() else "cpu"

train_set_noise = DasDataset(train_files_noise)
test_set_noise = DasDataset(test_files_noise)

In [6]:
tail_scores_path = Path ("data/noise_tail_scores.npy")
tail_scores_path.parent.mkdir(exist_ok = True)

if tail_scores_path.exists():
    tail_scores = np.load(tail_scores_path)

else:
    tail_scores = []
    for file, file_path in enumerate (train_files_noise):
        print(f"processing file {file+1} / {len(train_files_noise)} :") 
        f = h5py.File(file_path, "r")
        patches = f["patches"]
        num_patches = patches.shape[0]
        for index in tqdm(range(num_patches)):
            patch = np.array(patches[index])
            mu = np.median(patch)
            centered_patch = patch - mu
            tail_score = np.percentile(np.abs(centered_patch),99.9)
            tail_scores.append(tail_score)

    tail_scores = np.asarray(tail_scores, dtype = np.float32)
    np.save(tail_scores_path, tail_scores)

    print("Tail scores saved")
print("Tail scores shape:", tail_scores.shape)            

processing file 1 / 1 :


  0%|          | 0/16384 [00:00<?, ?it/s]

Tail scores saved
Tail scores shape: (16384,)


In [5]:
tail_scores_path = Path ("data/noise_tail_scores.npy")
tail_scores = np.load(tail_scores_path)

In [6]:
upper_threshold = np.percentile(tail_scores,80)
extreme_threshold = np.percentile(tail_scores,99)

normal_indices = np.where(tail_scores < upper_threshold) [0]
upper_indices = np.where((tail_scores >= upper_threshold) & (tail_scores < extreme_threshold))[0]
extreme_indices = np.where(tail_scores >= extreme_threshold)[0]
all_indices = np.arange(len(tail_scores))

print("all patches: ", len(all_indices))
print("normal patches: ", len(normal_indices))
print("upper tail patches: ", len(upper_indices))
print("extreme tail patches: ", len(extreme_indices))

all patches:  16384
normal patches:  13107
upper tail patches:  3113
extreme tail patches:  164


In [10]:
medians_path = Path ("data/noise_training_medians.npy")
medians_path.parent.mkdir(exist_ok = True)

if medians_path.exists():
    training_medians = np.load(medians_path)

else:
    training_medians = []
    for file, file_path in enumerate (train_files_noise):
        print(f"processing file {file+1} / {len(train_files_noise)} :") 
        f = h5py.File(file_path, "r")
        patches = f["patches"]
        num_patches = patches.shape[0]
        for index in tqdm(range(num_patches)):
            patch = np.array(patches[index])
            mu = np.median(patch)
            training_medians.append(mu)
    training_medians = np.asarray(training_medians, dtype = np.float32)
    np.save(medians_path, training_medians)

    print("training medians saved")
print("training medians shape:", training_medians.shape)            

processing file 1 / 1 :


  0%|          | 0/16384 [00:00<?, ?it/s]

training medians saved
training medians shape: (16384,)
